# Lab 6: Train and inspect a Diffusion Policy

**Goal:** Check the action windows, train-only statistics, checkpoint, and denoising settings for one Diffusion Policy. Then read its seeded development result and compare five and twenty denoising steps.

This lab uses one bowl-plate task. It does not use final states. It does not select the policy from final results.


## 1. Load the fixed recipe

Diffusion Policy learns to remove noise from an action sequence. During training, it receives an action window with added noise and learns to predict that noise. During a rollout, it starts from noise and repeats the denoising step. The result is a sequence of actions. The policy executes the declared part of that sequence, then uses new observations to plan again.

This course recipe uses two observations, a 16-step action horizon, and eight executed actions. The U-Net predicts all 16 action values at each denoising step. The first executed action is aligned with the current observation. The next observation window starts at the previous frame and ends at the current frame.


In [ ]:
import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import yaml

def find_root():
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the repository.")


WORKSPACE = find_root()
assert (WORKSPACE / "configs/train/diffusion_bowl_plate.yaml").is_file(), "The required course file is missing."
CONFIGS = WORKSPACE / "configs"
DATA_ROOT = WORKSPACE / ".local-data"
for cache_dir in (DATA_ROOT / "hf", DATA_ROOT / "hf/hub", DATA_ROOT / "hf/datasets"):
    cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(DATA_ROOT / "hf")
os.environ["HF_HUB_CACHE"] = str(DATA_ROOT / "hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(DATA_ROOT / "hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
REFERENCE = WORKSPACE / "manifests/diffusion_reference.yaml"
CHECKPOINT = DATA_ROOT / "runs/diffusion-bowl-plate-reference-2000/checkpoint-002000"
recipe = yaml.safe_load((CONFIGS / "train/diffusion_bowl_plate.yaml").read_text())
print({key: recipe[key] for key in ("task_alias", "seed", "steps", "batch_size", "train_episode_count", "n_obs_steps", "horizon", "n_action_steps", "do_mask_loss_for_padding", "development_state_ids")})


## 2. Check the observation and action windows

At 10 frames per second, an observation offset of `-0.1` seconds means the prior frame. Offset `0.0` means the current frame. The action window starts one frame before the current frame because it covers the same span as the observation window and then the future. The policy executes indices 1–8 from that 16-step window.

The loader drops the last seven frames of each training episode as current frames. It also marks any remaining padded action values. The loss mask excludes those padded values. The next cell checks the real training dataset. It does not use development or final data.


In [ ]:
from vla_lab.policy.diffusion import make_diffusion_config
from vla_lab.training.diffusion import (
    FPS,
    _frozen_diffusion_dataset,
    diffusion_delta_timestamps,
    eligible_frame_indices,
    executed_action_slice,
    validate_horizons,
)

policy_config = make_diffusion_config(device="cpu")
validate_horizons(policy_config)
deltas = diffusion_delta_timestamps(policy_config, FPS)
exec_start, exec_end = executed_action_slice(policy_config)
assert (policy_config.n_obs_steps, policy_config.horizon, policy_config.n_action_steps) == (2, 16, 8)
assert (exec_start, exec_end) == (1, 9)
assert recipe["do_mask_loss_for_padding"] is True
print("observation offsets (s):", deltas["observation.state"])
print("action offsets (s):", deltas["action"])
print("executed action indices:", list(range(exec_start, exec_end)))
print("drop_n_last_frames:", policy_config.drop_n_last_frames)

split_manifest = yaml.safe_load((CONFIGS / "data/episodes.yaml").read_text())
split = split_manifest["tasks"][recipe["task_alias"]]
dataset = _frozen_diffusion_dataset(policy_config, split, split_manifest["dataset_revision"], DATA_ROOT)
frame_table = dataset.hf_dataset
eligible = eligible_frame_indices(
    [int(x) for x in frame_table["episode_index"]],
    [int(x) for x in frame_table["frame_index"]],
    policy_config.drop_n_last_frames,
)
last_row_by_episode = {}
for row, episode_id in enumerate(frame_table["episode_index"]):
    last_row_by_episode[int(episode_id)] = row
last_eligible_by_episode = {}
for row in eligible:
    last_eligible_by_episode[int(frame_table[row]["episode_index"])] = row
assert len(eligible) == 3388, f"Expected 3,388 usable windows; got {len(eligible)}"
assert len(last_eligible_by_episode) == recipe["train_episode_count"]
print(f"training episodes: {len(last_eligible_by_episode)}; usable windows: {len(eligible)}")


In [ ]:
# Inspect the last source row and the last eligible row in one training episode.
episode_id = int(split["train_episode_ids"][0])
source_tail_row = last_row_by_episode[episode_id]
kept_tail_row = last_eligible_by_episode[episode_id]
source_tail = dataset[source_tail_row]
kept_tail = dataset[kept_tail_row]
source_pad = source_tail["action_is_pad"].bool()
kept_pad = kept_tail["action_is_pad"].bool()
assert source_pad.shape == (policy_config.horizon,)
assert kept_pad.shape == (policy_config.horizon,)
assert source_pad.any(), "The actual episode tail must show action padding."
print({
    "episode_id": episode_id,
    "episode_tail_padding_values": int(source_pad.sum()),
    "last_eligible_frame_padding_values": int(kept_pad.sum()),
    "eligible_tail_frame_index": int(frame_table[kept_tail_row]["frame_index"]),
    "episode_frame_count": int(frame_table["episode_index"].count(episode_id)),
})


## 3. Check visual and state statistics

The two images use fixed ImageNet channel mean and standard deviation. State and action values use minimum and maximum values computed from the 39 training episodes. The statistics file stores the split hash. A development state does not change these values.


In [ ]:
from vla_lab.policy.diffusion import IMAGENET_MEAN, IMAGENET_STD, make_diffusion_processors

preprocessor, postprocessor = make_diffusion_processors(policy_config, recipe["task_alias"], CONFIGS)
stats_file = CONFIGS / "data/stats/bowl_plate.json"
stats = json.loads(stats_file.read_text())
assert stats["split_sha256"] == split["split_sha256"]
assert stats["dataset_revision"] == split_manifest["dataset_revision"]
print("ImageNet mean:", IMAGENET_MEAN)
print("ImageNet std:", IMAGENET_STD)
for modality in ("observation.state", "action"):
    print(modality, "train-only stats SHA-256:", stats["modalities"][modality]["sha256"])
print("train split SHA-256:", split["split_sha256"])


## 4. Inspect training loss and time

The loss measures noise-prediction error on training windows. A lower value does not prove that the robot succeeds more often. The task result in Section 6 measures closed-loop behavior on separate development states.


In [ ]:
run_dir = DATA_ROOT / "runs/diffusion-bowl-plate-reference-2000"
run = json.loads((run_dir / "run.json").read_text())
metrics = [json.loads(line) for line in (run_dir / "metrics.jsonl").read_text().splitlines() if line]
assert run["steps"] == recipe["steps"] == 2000
assert run["train_episode_ids"] == split["train_episode_ids"]
assert len(metrics) == run["steps"]
fig, axis = plt.subplots(figsize=(9, 4))
axis.plot([row["step"] for row in metrics], [row["loss"] for row in metrics], linewidth=1)
axis.set(xlabel="Optimizer step", ylabel="Noise-prediction loss", title="Diffusion Policy training loss")
axis.grid(alpha=0.25)
plt.show()
print(f"steps: {run['steps']}; windows sampled: {run['sample_count']}; measured training loop: {metrics[-1]['elapsed_s']:.1f} s")
print(f"last recorded loss: {metrics[-1]['loss']:.6f}; device: {run['device']}")


## 5. Verify and reload the checkpoint

The bundle manifest links the model, config, processors, and training statistics to their SHA-256 hashes. The reload function checks these file hashes, rebuilds the policy on CPU, and checks the saved normalizer values.


In [ ]:
from vla_lab.training.diffusion import load_diffusion_bundle


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

bundle_manifest = json.loads((CHECKPOINT / "manifest.json").read_text())
assert bundle_manifest["model_sha256"] == sha256_file(CHECKPOINT / "model.safetensors")
assert bundle_manifest["split_sha256"] == split["split_sha256"]
assert bundle_manifest["n_obs_steps"] == 2 and bundle_manifest["horizon"] == 16
assert bundle_manifest["n_action_steps"] == 8
assert bundle_manifest["vision_weights_sha256"] == "f37072fd47e89c5e827621c5baffa7500819f7896bbacec160b1a16c560e07ec"
policy, saved_preprocessor, saved_postprocessor, loaded_manifest = load_diffusion_bundle(CHECKPOINT)
assert loaded_manifest == bundle_manifest
policy.eval()
print("model SHA-256:", bundle_manifest["model_sha256"])
print("config SHA-256:", bundle_manifest["config_sha256"])
print("ImageNet ResNet-18 weights SHA-256:", bundle_manifest["vision_weights_sha256"])
print("processor files:", sorted(bundle_manifest["processor_files_sha256"]))
print("strict local reload: passed")


## 6. Read the seeded development result

The reference manifest points to the saved 20-state result. Each state uses base seed `20260930` plus its state ID. Check every attempt, including failures. The final states 20–49 remain reserved.


In [ ]:
reference = yaml.safe_load(REFERENCE.read_text())["trained_reference"]
assert reference["checkpoint"] == str(CHECKPOINT.relative_to(WORKSPACE))
assert reference["model_sha256"] == bundle_manifest["model_sha256"]
assert reference["steps"] == run["steps"] == 2000
assert reference["train_episode_count"] == len(split["train_episode_ids"]) == 39
assert reference["base_seed"] == 20260930
assert reference["train_config_sha256"] == run["config_sha256"]
assert reference["development_successes"] == 15 and reference["development_attempts"] == 20

def artifact_path(value):
    path = Path(value)
    return path if path.is_absolute() else WORKSPACE / path

dev_dir = artifact_path(reference["development_result"])
dev_summary = json.loads((dev_dir / "summary.json").read_text())
dev_rows = [json.loads(line) for line in (dev_dir / "episodes.jsonl").read_text().splitlines() if line]
assert dev_summary["partition"] == "development"
assert dev_summary["state_ids"] == list(range(20))
assert dev_summary["attempts"] == len(dev_rows) == 20
assert all(row["partition"] == "development" for row in dev_rows)
assert not any(row["initial_state_id"] in range(20, 50) for row in dev_rows)
assert dev_summary["successes"] == sum(row["success"] for row in dev_rows)
assert dev_summary["policy_errors"] == sum(row["termination_reason"] == "policy_error" for row in dev_rows)
assert dev_summary["infrastructure_errors"] == sum(row["termination_reason"] == "infrastructure_error" for row in dev_rows)
print(f"seeded development result: {dev_summary['successes']}/{dev_summary['attempts']}")
print(f"policy errors: {dev_summary['policy_errors']}; infrastructure errors: {dev_summary['infrastructure_errors']}")


## 7. Compare five and twenty denoising steps

The tracked ablation uses paired development states 0–4. The reset seed and initial noise seed stay fixed for each state. Later noise draws can differ because the samplers consume different numbers of random values. The sampling-step count changes from 20 to 5. Compare each paired outcome and action-generation time. The weighted mean timing covers all sampler calls in the five live episodes, including the first cold call. A faster sampler can change behavior. Five paired states show an example; they do not give a stable success rate.


In [ ]:
ablation = json.loads(artifact_path(reference["sampler_ablation_result"]).read_text())
assert ablation["partition"] == "development"
assert ablation["state_ids"] == [0, 1, 2, 3, 4]
assert ablation["base_seed"] == reference["base_seed"]
assert ablation["model_sha256"] == bundle_manifest["model_sha256"]
assert ablation["selected_sampling_steps"] == 20

five = {row["state_id"]: row for row in ablation["sampling_steps"]["5"]["per_state"]}
twenty = {row["state_id"]: row for row in ablation["sampling_steps"]["20"]["per_state"]}
assert sorted(five) == sorted(twenty) == [0, 1, 2, 3, 4]
paired = []
for state_id in ablation["state_ids"]:
    paired.append({
        "state": state_id,
        "5-step success": five[state_id]["success"],
        "5-step sampler ms": five[state_id]["sampler_mean_ms"],
        "20-step success": twenty[state_id]["success"],
        "20-step sampler ms": twenty[state_id]["sampler_mean_ms"],
    })
print("weighted mean sampler latency (ms):")
print("5 steps:", ablation["sampling_steps"]["5"]["weighted_mean_sampler_ms"])
print("20 steps:", ablation["sampling_steps"]["20"]["weighted_mean_sampler_ms"])
print("20-step / 5-step latency ratio:", ablation["speed_ratio_20_over_5"])
print("success count:", ablation["sampling_steps"]["5"]["successes"], "/", ablation["sampling_steps"]["5"]["attempts"], "for 5 steps;",
      ablation["sampling_steps"]["20"]["successes"], "/", ablation["sampling_steps"]["20"]["attempts"], "for 20 steps")
for row in paired:
    print(row)
print("selection reason:", ablation["selection_reason"])


## 8. Run one live development state

This cell reloads the checkpoint and runs state 2 with reset seed `20260932`. State 2 succeeded in the saved reference run. State 0 failed and remains in the saved results. The cell uses the 20-step sampler. It writes a new result under `.local-data/runs`. It can take several minutes because it starts the simulator. It uses a development state only.


In [ ]:
import tempfile

from vla_lab.evaluation.diffusion import evaluate_diffusion_reference

live_output = Path(tempfile.mkdtemp(prefix="diffusion-live-dev-", dir=DATA_ROOT / "runs"))
live = evaluate_diffusion_reference(
    checkpoint=CHECKPOINT, configs_root=CONFIGS, data_root=DATA_ROOT,
    output_dir=live_output, state_ids=[2], seed=reference["base_seed"], inference_steps=20,
)
assert live["partition"] == "development" and live["state_ids"] == [2]
assert live["successes"] == 1, "Expected the saved state-2 success; inspect the new run"
print(json.dumps(live, indent=2))
print("live result directory:", live_output)


## Completion check

You are done when you can: name the two observation frames and action horizon; show how the loader handles episode-end padding; name the train-only state/action statistics and fixed visual statistics; verify and reload the checkpoint; report every seeded development attempt; compare the 5-step and 20-step sampler; and inspect one live development rollout.
